# 📚 Asistente de Investigación Multiagente
Este notebook permite cargar archivos PDF y hacer preguntas o tareas de investigación.
- Utiliza un sistema RAG con agentes para resumen, escritura e inferencia de referencias IEEE.
- Puedes cargar un archivo PDF y escribir un prompt de investigación.

In [1]:
# ✅ Cargar archivos necesarios
from langgraph_app.graph_builder import compiled_graph
from vector_storage.pdf_ingestor import ingest_pdf
import os

In [2]:
# ✅ Función principal
def ejecutar_asistente(prompt: str, pdf_path: str = None):
    if pdf_path:
        if not os.path.exists(pdf_path) or not pdf_path.endswith('.pdf'):
            print(f"❌ El archivo no existe o no es un PDF: {pdf_path}")
            return
        print(f"📄 Ingestando documento: {pdf_path}")
        ingest_pdf(pdf_path)
        print("✅ Ingesta completada.\n")

    print(f"📝 Prompt: {prompt}")
    print("🚀 Ejecutando agentes...\n")

    inputs = {"prompt": prompt}
    for output in compiled_graph.stream(inputs):
        for key, value in output.items():
            if key != "intermediate_steps":
                print(f"\n🎯 Resultado final:\n{value}")

## 📎 Subir archivo PDF desde el navegador

In [3]:
from IPython.display import display
from ipywidgets import FileUpload

uploader = FileUpload(accept='.pdf', multiple=False)
display(uploader)

FileUpload(value=(), accept='.pdf', description='Upload')

In [4]:
if uploader.value:
    for archivo in uploader.value:
        nombre = archivo['name']
        contenido = archivo['content']
        ruta = f"data/raw_papers/{nombre}"

        # Guardar archivo
        with open(ruta, "wb") as f:
            f.write(contenido)
        print(f"✅ Guardado: {ruta}")

✅ Guardado: data/raw_papers/LECT_Attention.pdf


## 🧪 Ejecutar el asistente

In [5]:
import sys
import os
import webbrowser

sys.path.append(os.path.abspath("."))  # Asegura que se encuentra el módulo

from main import ejecutar_asistente

# Llamada de ejemplo
result_path = "data/results"
os.makedirs(result_path, exist_ok=True)  # 🔧 Crea la carpeta si no existe

pdf_path = "data/raw_papers/LECT_Attention.pdf"
prompt = "Necesito un resumen de los puntos clave de todo este texto, tambien que me des las referencias y que me crees una introduccion para mi investigación"

respuesta = ejecutar_asistente(prompt, pdf_path)
resultado_texto = respuesta.get("result", "⚠️ No se obtuvo una respuesta.")

# Crear archivo .txt con mismo nombre base que el PDF
base_name = os.path.splitext(os.path.basename(pdf_path))[0]
output_path = os.path.join(result_path, f"{base_name}_resultado.txt")

with open(output_path, "w", encoding="utf-8") as file:
    file.write(resultado_texto)

# Abrir el archivo (en bloc de notas o visor predeterminado)
abs_path = os.path.abspath(output_path)
webbrowser.open(f"file://{abs_path}")

print(f"✅ Resultado guardado y abierto: {output_path}")


📄 Ingestando PDF: data/raw_papers/LECT_Attention.pdf
Directorios removidos
📂 El PDF ya está en data/raw_papers/LECT_Attention.pdf, no se copia.
📄 Cargadas 15 páginas del PDF.
✂️ Fragmentados en 52 fragmentos de texto.
📦 Vector store creado desde cero.
✅ Vector store actualizado con éxito.
✅ PDF procesado.

🧠 Ejecutando agentes con el prompt:
Necesito un resumen de los puntos clave de todo este texto, tambien que me des las referencias y que me crees una introduccion para mi investigación

📌 Nodo DECIDER (LLM) seleccionó: ['resumen', 'referencias', 'escritura']
🔹 Nodo RESUMEN ejecutado
🟡 Nodo REFERENCIAS ejecutado
🟩 Nodo ESCRITURA ejecutado
✅ Resultado guardado y abierto: data/results/LECT_Attention_resultado.txt


In [6]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
import pandas as pd

# Cargar el vector store
vector_store_path = "data/processed"
vector_store = FAISS.load_local(vector_store_path, OpenAIEmbeddings(), allow_dangerous_deserialization=True)

# Extraer los fragmentos y sus metadatos
fragmentos = []
for key, doc in vector_store.docstore._dict.items():
    fragmentos.append({
        "ID": key,
        "Contenido": doc.page_content,
        "Fuente": doc.metadata.get("source", ""),
        "Página": doc.metadata.get("page", "")
    })

# Mostrar los datos
df_fragmentos = pd.DataFrame(fragmentos)
df_fragmentos.head()  # Muestra los primeros fragmentos


,ID,Contenido,Fuente,Página
0,4885a0f5-e18f-4b02-b416-65373b79efd0,"Provided proper attribution is provided, Googl...",data/raw_papers/LECT_Attention.pdf,0
1,e0a5f689-7cfd-40c9-a45a-c13a9c655add,mechanism. We propose a new simple network arc...,data/raw_papers/LECT_Attention.pdf,0
2,e8221d4b-0541-4e0f-93ed-bc1a3614cd2f,best models from the literature. We show that ...,data/raw_papers/LECT_Attention.pdf,0
3,94ff4515-affd-46a1-b838-be54ad60dd65,efficient inference and visualizations. Lukasz...,data/raw_papers/LECT_Attention.pdf,0
4,67de2cb9-ba65-4d4e-b8dd-7be35a98f78a,"1 Introduction\nRecurrent neural networks, lon...",data/raw_papers/LECT_Attention.pdf,1


In [7]:
df_fragmentos["Fuente"].value_counts()


Fuente
data/raw_papers/LECT_Attention.pdf    52
Name: count, dtype: int64

In [8]:
from PyPDF2 import PdfReader

reader = PdfReader("data/raw_papers/public_Scrum-Guide-US.pdf")
paginas_con_texto = [i for i, p in enumerate(reader.pages) if p.extract_text().strip()]
print(f"Páginas con texto: {len(paginas_con_texto)} de {len(reader.pages)}")


Páginas con texto: 16 de 16
